In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory

def process_all_pdf(pdf_directory):
    """Process all the pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf diles recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n processing :{pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source infor to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            print(f"all documetn now-> {len(all_documents)}")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\n Total Documetns loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdf("../data")

Found 3 PDF files to process

 processing :gs statement uwa.pdf
Loaded 2 pages
all documetn now-> 2

 processing :Sanjana_SOP_UWA_MIT_v4.pdf
Loaded 7 pages
all documetn now-> 9

 processing :Hasibul_Hoque_Dependent_SOP (1).pdf
Loaded 3 pages
all documetn now-> 12

 Total Documetns loaded: 12
1. I, Sanjana Akter Roshni, reside in Dhaka, Bangladesh, with my husband, who is 
employed as a Software Engineer. My father owns and manages Sydney Homes Ltd, a 
real estate company specializing in residential apartment construction and sales. I am not 
currently involved in any community or NGO leadership roles. 
My primary sponsor is my father, with fixed deposit of BDT 50,00,000 (approximately 
AUD 57,145). My secondary sponsor is my mother-in-law, with fixed deposit of BDT 
40,00,000 (approximately AUD 45,716), making a combined total of BDT 90,00,000 
(approximately AUD 102,861). My father’s annual income is approximately BDT 
35,50,000. He owns a real estate company and land valued at approx

In [22]:
output_file = "../data/text_files/combined_documents.txt"

with open(output_file,"w",encoding="utf-8") as f:
    f.write(all_pdf_documents[0].page_content)
    f.write(all_pdf_documents[1].page_content)
    

In [24]:
### Text splitting

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )

    split_doc = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_doc)} chunks")

    #show example of chunk
    if split_doc:
        print(f"\n Example chunk:")
        print(f"content: {split_doc[0].page_content[:200]}...")
        print(f"metadata: {split_doc[0].metadata}")

    return split_doc

In [25]:
chunks = split_documents(all_pdf_documents)
chunks

Split 12 documents into 37 chunks

 Example chunk:
content: 1. I, Sanjana Akter Roshni, reside in Dhaka, Bangladesh, with my husband, who is 
employed as a Software Engineer. My father owns and manages Sydney Homes Ltd, a 
real estate company specializing in r...
metadata: {'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': '', 'creationdate': "D:20260711172409Z00'00'", 'source': '../data/pdf/gs statement uwa.pdf', 'file_path': '../data/pdf/gs statement uwa.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260711172409Z00'00'", 'trapped': '', 'modDate': "D:20260711172409Z00'00'", 'creationDate': "D:20260711172409Z00'00'", 'page': 0, 'source_file': 'gs statement uwa.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': '', 'creationdate': "D:20260711172409Z00'00'", 'source': '../data/pdf/gs statement uwa.pdf', 'file_path': '../data/pdf/gs statement uwa.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260711172409Z00'00'", 'trapped': '', 'modDate': "D:20260711172409Z00'00'", 'creationDate': "D:20260711172409Z00'00'", 'page': 0, 'source_file': 'gs statement uwa.pdf', 'file_type': 'pdf'}, page_content='1. I, Sanjana Akter Roshni, reside in Dhaka, Bangladesh, with my husband, who is \nemployed as a Software Engineer. My father owns and manages Sydney Homes Ltd, a \nreal estate company specializing in residential apartment construction and sales. I am not \ncurrently involved in any community or NGO leadership roles. \nMy primary sponsor is my father, with fixed deposit of BDT 50,00,000 (approximately \nAUD 57,145). My secondary sponsor is

In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [27]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name: str = "all-MiniLM-L6-V2"):
        """ 
        Initialize the embedding manager

        Args: model_name: HuggingFace model name for sentense embeddings 
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """ 
        Generate embedding for a list of texts

        Args: 
            texts: List of text strings to embed
        Returns:
            numpy array of embedding with shape (len(texts),embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddinf for {len(texts)} texts ...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 23627.94it/s]


Model loaded successfully. Embedding dimension: 384


In [29]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    
    def _initialize_store(self):
        """Initialize chromaDB client and collection"""

        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description" : "pdf document embedding for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error initialize vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any],embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection : 0


In [31]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks,embeddings)

Generating embeddinf for 37 texts ...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.66it/s]


Generated embeddings with shape (37, 384)
Adding 37 documents to vector store...
Successfully added 37 documents to vector store
Total documents in collection: 37
